In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
!nvidia-smi

Tue Oct 13 04:50:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
!pip install --upgrade langchain
!pip install langchain_community
!pip install langchain_text_splitter
!pip install langchain_chroma
!pip install langchain_groq
!pip install langchain

ERROR: Could not find a version that satisfies the requirement langchain_text_splitter (from versions: none)
ERROR: No matching distribution found for langchain_text_splitter


In [18]:
!pip install langchain-huggingface langchain
import os
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "api key"

In [20]:
# Fetch the PDF from the URL
import requests
url = "https://rbberger.github.io/assets/files/smr2574/OOPConcepts.pdf"
response = requests.get(url)

In [21]:
with open("python_inbuildfunction.pdf","wb") as f:
  f.write(response.content)

In [22]:
from langchain_community.document_loaders import UnstructuredFileLoader

loader = UnstructuredFileLoader("python_inbuildfunction.pdf")

In [23]:
!pip install unstructured

In [24]:
loader

In [25]:
document = loader.load()
document

[Document(metadata={'source': 'python_inbuildfunction.pdf'}, page_content='Concepts of Object-Oriented Programming\n\nRichard Berger\n\nrichard.berger@jku.at Johannes Kepler University Linz\n\nWhat this talk is about\n\nIntroduction to Objects & Classes • Building objects • Composition • Encapsulation • Inheritance • Polymorphism\n\nBest practices\n\nRecommendations • Design Patterns\n\nConclusion\n\nPro & Cons of OOP\n\nWhat is Object-Oriented Programming?\n\nOOP started in 1960s (Simula, SmallTalk) • Using objects as basic unit of computation • Allows to extend type system\n\nUsage: just like basic types (int, double, float, char, …) • But with own structure and behavior\n\nStatic Languages (C++)\n\nTypes are known at compile time\n\nDynamic Languages (Python)\n\nTypes can be manipulated at runtime\n\nmessage\n\nObject\n\nmessage\n\nObject\n\nmessage\n\nObject\n\nI. Objects & Classes\n\nDefining new objects, object lifetime\n\nWhere are these “objects”?\n\nObjects exist in memory at 

In [26]:
!pip install "unstructured[pdf]"

In [27]:
text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
texts = text_splitter.split_documents(document)

In [28]:
type(texts)

list

In [29]:
len(texts)

13

In [30]:
texts[6]

Document(metadata={'source': 'python_inbuildfunction.pdf'}, page_content='a->fullThrottle();\n\n// NEW! will call Boeing747::fullThrottle\n\nb->fullThrottle();\n\n// NEW! will call Cessna206h::fullThrottle\n\nc->fullThrottle();\n\nObjects of derived classes can be used as base class objects.\n\nPolymorphism allows us to modify the behavior of base classes and replacing the implementation of methods. • Polymorphic methods must be\n\ndeclared virtual (in C++)\n\nPolymorphism\n\n// store objects of different types in a // datastructure using its base class Airplane ** airplanes = new Airplane*[10]; ... for(int i = 0; i < 10; i++) {\n\n// will choose correct implementation // dynamically at runtime airplanes[i]->fullThrottle();\n\n}\n\nvoid foo(Airplane & a) {\n\n// this function will work with // any class derived from Airplane\n\n}\n\nUse base class to implement general algorithms and data structures which work with any derived type\n\nDynamic dispatch determines the type of an object at

In [31]:
embeddings = HuggingFaceEmbeddings()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [77]:
persist_directory = "vector_db"

In [78]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [79]:
Retriever = vectordb.as_retriever()

In [80]:
# llm from groq
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=1
)

In [81]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=Retriever,
    return_source_documents=True
)

In [83]:
# invoke the qa chain and get a response for user query (requesting a brief summary to avoid rate limits)
query = "can you summarize the topic from the pdf in a brief 3-sentence summary?"
response = qa_chain.invoke({"query": query})

In [84]:
print(response)

{'query': 'can you summarize the topic from the pdf in a brief 3-sentence summary?', 'result': 'This presentation introduces the fundamental concepts of Object-Oriented Programming (OOP), explaining that objects are instances of classes which define both the memory structure and the behavior of these entities. It details the object lifecycle, distinguishing between static (stack) and dynamic (heap) allocation, as well as the role of constructors in initializing object state. Finally, the talk covers core OOP principles such as composition, encapsulation, inheritance, and polymorphism, alongside best practices and design patterns.', 'source_documents': [Document(id='e49ca10a-fc5b-4070-b8fe-1afe35aa3d99', metadata={'source': 'python_inbuildfunction.pdf'}, page_content='Structure and Behavior of similar objects is defined by their class. An object is also called an instance of a class.\n\nClasses\n\nclass Vector2D { public:\n\ndouble x; double y;\n\ndouble length() {\n\nreturn sqrt(x*x + 

In [85]:
print(response["result"])

This presentation introduces the fundamental concepts of Object-Oriented Programming (OOP), explaining that objects are instances of classes which define both the memory structure and the behavior of these entities. It details the object lifecycle, distinguishing between static (stack) and dynamic (heap) allocation, as well as the role of constructors in initializing object state. Finally, the talk covers core OOP principles such as composition, encapsulation, inheritance, and polymorphism, alongside best practices and design patterns.


In [86]:
# invoke the qa chain and get a response for user query
query = "Give me summary of all function from this pdf?"
response = qa_chain.invoke({"query": query})
print(response["result"])
print("*"*30)
print("Source Document:", response["source_documents"][0].metadata["source"])

Based on the provided text, here is a summary of all the functions (member methods and static methods) defined within the code examples:

**1. Methods in class `Vector2D`:**

*   **`double length()`**: A member method that calculates and returns the length of the vector using the formula $\sqrt{x^2 + y^2}$.
*   **`Vector2D()`** (Default Constructor): A constructor that initializes an instance of `Vector20` by setting both member variables `x` and `y` to `0.0`.
*   **`Vector2D(double x1, double y1)`** (Parameterized Constructor): A constructor that initializes an instance of `Vector2D` by setting `x` and `y` to the values provided as arguments (`x1` and `y1`).

**2. Methods in class `Point2D`:**

*   **`Point2D(double x1, double y1)`** (Constructor): A constructor that initializes a new point. It assigns the current value of the static variable `numPoints` to the instance's `identifier`, then increments `numPoints`. It also sets `x` and `y` to the arguments `x1` and `y1`.
*   **`static 

In [87]:
 !pip install gradio langchain langchain-community langchain-openai
 !pip install langchain-text-splitters pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.1/146.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 87.0 MB/s eta 0:00:00


In [88]:
import gradio as gr

def rag_chat(message, history):
    try:
        # Invoke RAG chain
        response = qa_chain.invoke({
            "query": message
        })

        # Get answer
        answer = response["result"]

        # Get source documents
        source_documents = response.get("source_documents", [])

        # Prepare source information
        sources = []

        for doc in source_documents:
            source = doc.metadata.get("source", "Unknown")
            page = doc.metadata.get("page", "N/A")

            source_info = f"{source} - Page: {page}"

            if source_info not in sources:
                sources.append(source_info)

        # Add sources to final answer
        if sources:
            answer += "\n\n### Sources\n"

            for source in sources:
                answer += f"- {source}\n"

        return answer

    except Exception as e:
        return f"Error: {str(e)}"

demo = gr.ChatInterface(
    fn=rag_chat,
    title="📚 PDF RAG Assistant",
    description="Ask questions about your PDF document.",
    examples=[
        "Give me a summary of this PDF.",
        "What are the main topics in this document?",
        "Give me summary of all functions from this PDF.",
        "Explain the important concepts."
    ]
)


# --------------------------------------------------
# Launch Application
# --------------------------------------------------

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ec0426a4c4c442402.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [89]:
!pip install langchain_openai

In [90]:
import gradio as gr

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA


# ============================================================
# Global QA Chain
# ============================================================

qa_chain = None


# ============================================================
# 1. Upload PDF and Create RAG
# ============================================================

def process_pdf(pdf_file):

    global qa_chain

    if pdf_file is None:
        return "❌ Please upload a PDF file."

    try:
        # -------------------------------------------
        # Load PDF
        # -------------------------------------------
        loader = PyPDFLoader(pdf_file.name)
        documents = loader.load()

        # -------------------------------------------
        # Split PDF
        # -------------------------------------------
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

        chunks = text_splitter.split_documents(documents)

        # -------------------------------------------
        # Embeddings
        # -------------------------------------------
        embeddings = OpenAIEmbeddings(
            model="text-embedding-3-small"
        )

        # -------------------------------------------
        # Vector Database
        # -------------------------------------------
        vectorstore = FAISS.from_documents(
            chunks,
            embeddings
        )

        # -------------------------------------------
        # Retriever
        # -------------------------------------------
        retriever = vectorstore.as_retriever(
            search_kwargs={"k": 5}
        )

        # -------------------------------------------
        # LLM
        # -------------------------------------------
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        )

        # -------------------------------------------
        # RAG Chain
        # -------------------------------------------
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True
        )

        return f"""
### ✅ PDF Ready!

**Pages:** {len(documents)}
**Chunks:** {len(chunks)}

You can now ask questions about your PDF.
"""

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ============================================================
# 2. Ask Question
# ============================================================

def ask_pdf(question):

    global qa_chain

    if qa_chain is None:
        return "⚠️ Please upload and process a PDF first."

    if not question.strip():
        return "Please enter a question."

    try:

        response = qa_chain.invoke({
            "query": question
        })

        answer = response["result"]

        # -------------------------------------------
        # Get Sources
        # -------------------------------------------

        source_documents = response.get(
            "source_documents",
            []
        )

        pages = []

        for doc in source_documents:

            page = doc.metadata.get("page")

            if page is not None:

                # PyPDF page starts from 0
                page = page + 1

                if page not in pages:
                    pages.append(page)

        # -------------------------------------------
        # Add page references
        # -------------------------------------------

        if pages:

            pages = sorted(pages)

            answer += "\n\n---\n"

            answer += "📄 **Source Pages:** "

            answer += ", ".join(
                [str(page) for page in pages]
            )

        return answer

    except Exception as e:

        return f"❌ Error: {str(e)}"


# ============================================================
# 3. Gradio UI
# ============================================================

with gr.Blocks(title="PDF RAG Assistant") as demo:

    gr.Markdown(
        """
        # 📚 PDF RAG Assistant

        **Upload any PDF → Process PDF → Ask Questions**
        """
    )

    # --------------------------------------------------------
    # PDF Upload
    # --------------------------------------------------------

    pdf_file = gr.File(
        label="📄 Upload PDF",
        file_types=[".pdf"]
    )

    process_button = gr.Button(
        "🚀 Process PDF"
    )

    status = gr.Markdown()

    process_button.click(
        fn=process_pdf,
        inputs=pdf_file,
        outputs=status
    )

    gr.Markdown("---")

    # --------------------------------------------------------
    # Question
    # --------------------------------------------------------

    question = gr.Textbox(
        label="💬 Ask a Question",
        placeholder="Example: Give me a summary of this PDF"
    )

    ask_button = gr.Button(
        "🔍 Ask PDF"
    )

    answer = gr.Markdown(
        label="Answer"
    )

    ask_button.click(
        fn=ask_pdf,
        inputs=question,
        outputs=answer
    )

    # Press ENTER
    question.submit(
        fn=ask_pdf,
        inputs=question,
        outputs=answer
    )


# ============================================================
# 4. Launch
# ============================================================

if __name__ == "__main__":

    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e1e1235eaa0c7ba10c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [91]:
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

# ============================================================
# Global QA Chain
# ============================================================
qa_chain = None


# ============================================================
# 1. Upload & Process PDF
# ============================================================
def process_pdf(pdf_file):
    global qa_chain

    if pdf_file is None:
        return "❌ Please upload a PDF file."

    try:
        # Load PDF
        loader = PyPDFLoader(pdf_file.name)
        documents = loader.load()

        if not documents:
            return "❌ The uploaded PDF contains no readable text."

        # Split into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )
        chunks = text_splitter.split_documents(documents)

        # Generate Embeddings
        embeddings = OpenAIEmbeddings(
            model="text-embedding-3-small"
        )

        # Index in FAISS Vector Store
        vectorstore = FAISS.from_documents(chunks, embeddings)

        # Setup Retriever
        retriever = vectorstore.as_retriever(
            search_kwargs={"k": 5}
        )

        # Initialize LLM
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        )

        # Create RAG Chain
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True
        )

        return f"""### ✅ PDF Ready!
- **Pages Processed:** {len(documents)}
- **Text Chunks:** {len(chunks)}

You can now start asking questions in the chat!"""

    except Exception as e:
        return f"❌ Error processing PDF: {str(e)}"


# ============================================================
# 2. Chat Function for gr.ChatInterface
# ============================================================
def rag_chat(message, history):
    global qa_chain

    if qa_chain is None:
        return "⚠️ Please upload and process a PDF first before asking questions."

    if not message.strip():
        return "Please enter a valid query."

    try:
        # Invoke RAG chain
        response = qa_chain.invoke({"query": message})
        answer = response.get("result", "")

        # Extract source documents and pages
        source_documents = response.get("source_documents", [])
        pages = []

        for doc in source_documents:
            page = doc.metadata.get("page")
            if page is not None:
                # PyPDF uses 0-based page index, convert to 1-based
                page_num = page + 1
                if page_num not in pages:
                    pages.append(page_num)

        # Append source pages to answer
        if pages:
            pages = sorted(pages)
            answer += "\n\n---\n**📄 Source Pages:** " + ", ".join(str(p) for p in pages)

        return answer

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ============================================================
# 3. Gradio UI Layout
# ============================================================
with gr.Blocks(title="📚 PDF RAG Assistant") as demo:
    gr.Markdown("# 📚 Conversational PDF RAG Assistant")
    gr.Markdown("Upload a PDF document, click **Process PDF**, and ask questions via the chat interface.")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_file = gr.File(
                label="📄 Upload PDF Document",
                file_types=[".pdf"],
                type="filepath"
            )
            process_btn = gr.Button("🚀 Process PDF", variant="primary")
            status_output = gr.Markdown()

            # Trigger processing
            process_btn.click(
                fn=process_pdf,
                inputs=[pdf_file],
                outputs=[status_output]
            )

        with gr.Column(scale=2):
            chat_ui = gr.ChatInterface(
                fn=rag_chat,
                examples=[
                    "Give me a summary of this PDF.",
                    "What are the main topics in this document?",
                    "Explain the key concepts discussed."
                ]
            )

# ============================================================
# 4. Launch Application
# ============================================================
if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2b710eb3651fa06ee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
